# Network Classification and Link Prediction Analysis

This notebook reproduces **Figure 4**, **Figure 6**, **Table 2**, and **Table 3** from Sections 4.3, 4.4, and 4.5, which validate the relationship between multiscale entropy and network predictability.

## Overview

We perform two complementary analyses on the ICON dataset:

1. **Clustering Analysis (Figure 4, Table 2)**: K-means clustering (k=3) on five-dimensional entropy vectors [100%, 80%, 60%, 40%, 20%] reveals three groups aligned with entropy behaviors: hybrid (social), increasing (economic/technological), and stable (transportation/informational).

2. **Link Prediction Analysis (Figure 6, Table 3)**: We compute link prediction entropy using Jaccard and Adamic-Adar indices across reduction levels, demonstrating that structural compression entropy strongly correlates with network predictability across domains.

Together, these analyses establish multiscale entropy as both a classification tool and a predictor of link prediction performance.

## Imports and Setup

In [ ]:
pip install pandas==1.5.3
pip install sklearn

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

/tmp/ipykernel_1441/912229180.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


In [ ]:
import os
import sys

##get current file directory
current_dir = os.path.dirname(os.path.abspath('__file__'))
##get the parent directory
parent_dir = os.path.dirname(current_dir)
# Add the src directory to sys.path
src_dir = os.path.join(parent_dir, '../')
sys.path.append(src_dir)

from algorithm.calculo_entropia import *
import algorithm.calculo_entropia

from algorithm.coarsening_utils import *
import algorithm.graph_utils
import algorithm.coarsening_utils as cu
from algorithm.coarsening_utils import plot_coarsening_vertical

import numpy as np
import scipy as sp

import matplotlib
import matplotlib.pylab as plt
from mpl_toolkits.mplot3d import Axes3D

import networkx as nx
import pygsp as gsp
from pygsp import graphs
gsp.plotting.BACKEND = 'matplotlib'

import pickle

def save_graphs(graph_dict, filename):
    """
    Save the dictionary of graphs to a file.
    """
    # Convert PyGSP graphs to NetworkX graphs for easier serialization
    nx_graph_dict = {
        size: [nx.from_scipy_sparse_array(g.W) for g in graphs]
        for size, graphs in graph_dict.items()
    }
    
    with open(filename, 'wb') as f:
        pickle.dump(nx_graph_dict, f)
    print(f"Graphs saved to {filename}")

def load_graphs(filename):
    """
    Load the dictionary of graphs from a file.
    """
    with open(filename, 'rb') as f:
        nx_graph_dict = pickle.load(f)
    print(f"Graphs loaded from {filename}")
    return nx_graph_dict

In [ ]:
import pandas as pd
print(pd.__version__)

1.5.3


In [ ]:
import pickle  
# load the data 
infile = open('./CommunityFitNet_updated.pickle','rb')  
df = pickle.load(infile) 

In [ ]:
# read edge lists for all networks
df_edgelists = df['edges_id'] # column 'edges_id' in dataframe df includes the edge list 
                              # for each network 
 
# extract the edge list for the first network 
edges_orig = df_edgelists.iloc[0] # a numpy array of edge list for original graph 


## Network Classification via Multiscale Entropy Clustering

To quantitatively validate the distinct entropy trajectory patterns observed across network domains, we perform unsupervised clustering using K-means (k=3) on the five-dimensional entropy vectors derived from each network's multiscale reduction sequence.

### Methodology

Each network is represented by a feature vector containing normalized compression entropy values at five reduction levels: [100%, 80%, 60%, 40%, 20%]. These vectors are standardized using z-score normalization before applying K-means clustering with k=3, chosen to align with the three dominant behavioral patterns identified earlier: **stable**, **increasing**, and **hybrid** entropy trajectories.

For visualization, we project the high-dimensional feature space onto its first two principal components using PCA. The clustering reveals clear spatial separation between groups, confirming that networks with similar multiscale entropy profiles naturally group together.

The results demonstrate a strong correspondence between network domain and cluster assignment. Social networks predominantly form one cluster (hybrid behavior), economic and technological networks another (increasing entropy), while transportation and informational networks group separately (stable entropy). Notably, biological networks are distributed across all clusters, reflecting the structural heterogeneity of biological systems.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns
from collections import defaultdict
from sklearn.decomposition import PCA

# Cargar los datos
with open("graph_families_analysis.json", "r") as file:
    data = json.load(file)

# Extraer los valores de entropía normalizada de length compression
graphs = []  # Lista de nombres de grafos
features = []  # Lista para almacenar los vectores de 5 dimensiones
families = []

for family, graphs_data in data.items():
    for graph_name, graph_info in graphs_data.items():
        if "reductions" in graph_info:
            reductions = graph_info["reductions"]
            if all(str(p) in reductions for p in ["100", "80", "60", "40", "20"]):
                entropy_values = [
                    reductions[str(p)]["entropy_arithmetic"]["normalized"]
                    for p in [100, 80, 60, 40, 20]
                ]
                features.append(entropy_values)
                graphs.append(graph_name)
                families.append(family)

# Convertir a numpy array
X = np.array(features)

# Estandarizar los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar KMeans con 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_scaled)
labels = kmeans.labels_
cluster_centers = kmeans.cluster_centers_

# Visualizar los clusters usando PCA para reducción a 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
centers_pca = pca.transform(cluster_centers)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=families, palette="tab10", s=100, alpha=0.8)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c="red", marker="x", s=200, label="Cluster Center")

# Dibujar círculos alrededor de los clusters
for center in centers_pca:
    plt.gca().add_patch(plt.Circle(center, 0.5, color='gray', fill=False, linestyle='dashed'))

plt.title("Graph Clustering by Length Compression Normalized Entropy (KMeans, k=3)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(title="Family Domain")
plt.show()

# Agrupar datos por cluster
cluster_composition = defaultdict(lambda: defaultdict(int))
for i, (family, graph_name) in enumerate(zip(families, graphs)):
    cluster_composition[labels[i]][family] += 1

# Mostrar la composición de los clusters
print("Composición de los clusters:\n")
for cluster, family_counts in sorted(cluster_composition.items()):
    print(f"Cluster {cluster}:")
    for family, count in family_counts.items():
        print(f"{family}: {count} grafos")
    print("")


## Link Prediction Analysis

**For the following analysis we used the "graph_families_multi_entropy_analysis.json" file which contained the information of each entropy in every level of reduction of every graph. All the information was obtained from the experiments that appear in the other sections of this repository.**

To quantify the relationship between multiscale structural entropy and link prediction
entropy, we evaluated how well the latter can be predicted from the former using
regression models. This extends prior work by Sun et al. [8], which demonstrated a
strong linear relationship between structural complexity and predictability at a single
scale.

We trained five linear regression models using entropy at one (Model 1) to five
(Model 5) levels of node reduction (100% to 20%). As shown in Table 3, including 
additional entropy scales substantially improves predictive performance.

In [ ]:
import json
import pandas as pd

# Load JSON file
with open("graph_families_multi_entropy_analysis.json", "r") as f:
    data = json.load(f)

# Collect rows into a list of dicts
rows = []
for domain, graphs in data.items():
    for graph_name, graph_info in graphs.items():
        for reduction_key, reduction_data in graph_info["reductions"].items():
            row = {
                "Domain": domain,
                "Graph_Name": graph_name,
                "Graph_Portion": reduction_data.get("graph_portion"),
                "Number_Nodes": reduction_data.get("number_nodes"),
                "Number_Edges": reduction_data.get("number_edges"),
                "Average_Degree": reduction_data.get("ave_degree"),
                "Entropy_Arithmetic_Graph": reduction_data["entropy_arithmetic"]["graph"],
                "Entropy_Arithmetic_Random": reduction_data["entropy_arithmetic"]["random"],
                "Entropy_Arithmetic_Normalized": reduction_data["entropy_arithmetic"]["normalized"],
                "Entropy_Jaccard_Graph": reduction_data["entropy_linkPrediction_Jaccard"]["graph"],
                "Entropy_Jaccard_Random": reduction_data["entropy_linkPrediction_Jaccard"]["random"],
                "Entropy_Jaccard_Normalized": reduction_data["entropy_linkPrediction_Jaccard"]["normalized"],
                "Entropy_AdamicAdar_Graph": reduction_data["entropy_linkPrediction_AdamicAdar"]["graph"],
                "Entropy_AdamicAdar_Random": reduction_data["entropy_linkPrediction_AdamicAdar"]["random"],
                "Entropy_AdamicAdar_Normalized": reduction_data["entropy_linkPrediction_AdamicAdar"]["normalized"]
            }
            rows.append(row)

# Create DataFrame
df = pd.DataFrame(rows)

# Show the first few rows
print(df.head())


In [ ]:
import pandas as pd
import statsmodels.api as sm

# Define all subsets of portions
portion_subsets = [
    [100],
    [100, 80],
    [100, 80, 60],
    [100, 80, 60, 40],
    [100, 80, 60, 40, 20]
]

results = []

# Build model dataset
df_wide = df[df["Graph_Portion"].isin([100, 80, 60, 40, 20])][
    ["Graph_Name", "Graph_Portion", "Entropy_Arithmetic_Normalized"]
].pivot(index="Graph_Name", columns="Graph_Portion", values="Entropy_Arithmetic_Normalized")

df_target = df[df["Graph_Portion"] == 100][["Graph_Name", "Entropy_AdamicAdar_Normalized"]].set_index("Graph_Name")
df_model = df_wide.merge(df_target, left_index=True, right_index=True)

# Fit models with increasing number of predictors
for portions in portion_subsets:
    X = df_model[portions].copy()
    X.columns = [f"Entropy_Arithmetic_{p}" for p in portions]
    X = sm.add_constant(X)
    y = df_model["Entropy_AdamicAdar_Normalized"]

    model = sm.OLS(y, X).fit()

    for var_name, coef, pval in zip(model.params.index, model.params.values, model.pvalues.values):
        results.append({
            "Model": f"{portions}",
            "Variable": var_name,
            "Coefficient": coef,
            "P-Value": pval,
            "R²": model.rsquared,
            "R² Adjusted": model.rsquared_adj,
            "No. Observations": int(model.nobs),
            "Prob (F-statistic)": model.f_pvalue
        })

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df = results_df[
    ["Model", "Variable", "No. Observations", "Coefficient", "P-Value", "Prob (F-statistic)", "R²", "R² Adjusted"]
]

# Round numeric values for better readability
results_df[["Coefficient", "P-Value", "Prob (F-statistic)", "R²", "R² Adjusted"]] = results_df[
    ["Coefficient", "P-Value", "Prob (F-statistic)", "R²", "R² Adjusted"]
].round(5)

# Display
print(results_df.to_string(index=False))


### Figure 6: Model Comparison - Predicted vs Actual Entropy

Figure 6 further illustrates the predictive gain obtained by incorporating multiscale 
entropy features. Model 1 (left) shows moderate alignment between the predicted and 
actual values, but there is substantial variance, particularly for social and economic 
networks, which often lie far from the diagonal. In contrast, Model 5 (right), which 
integrates entropy at five reduction levels, achieves notably tighter alignment across 
all domains. The concentration of points near the identity line and the reduction in 
domain-specific dispersion demonstrate the improved fit and generalizability of the 
multiscale approach, especially for biological and transportation networks.

**Fig. 6:** Predicted vs. actual values for normalized Adamic-Adar entropy using two 
regression models: Model 1 (left), which uses only entropy at 100% graph portion, and 
Model 5 (right), which includes entropy at 100%, 80%, 60%, 40%, and 20%. Colors 
represent different network domains. The diagonal dashed line represents perfect 
prediction.

In [ ]:
import statsmodels.api as sm
from scipy.stats import f

# Define the two sets of predictors
model_1_portions = [100]
model_5_portions = [100, 80, 60, 40, 20]

# Prepare X and y
X1 = df_model[model_1_portions].copy()
X1.columns = [f"Entropy_Arithmetic_{p}" for p in model_1_portions]
X1 = sm.add_constant(X1)

X5 = df_model[model_5_portions].copy()
X5.columns = [f"Entropy_Arithmetic_{p}" for p in model_5_portions]
X5 = sm.add_constant(X5)

y = df_model["Entropy_AdamicAdar_Normalized"]

# Fit models
model1 = sm.OLS(y, X1).fit()
model5 = sm.OLS(y, X5).fit()

# Extract RSS and df
RSS1 = model1.ssr
RSS5 = model5.ssr
df1 = model1.df_resid
df5 = model5.df_resid
dof_num = df1 - df5
dof_den = df5

# Compute F-statistic
F_stat = ((RSS1 - RSS5) / dof_num) / (RSS5 / dof_den)

# Compute p-value
p_value = f.sf(F_stat, dof_num, dof_den)

# Report
print(f"F-statistic (Model 1 vs Model 5): {F_stat:.4f}")
print(f"p-value: {p_value:.4g}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Prepare data: pivot and merge with Domain info
df_wide = df[df["Graph_Portion"].isin([100, 80, 60, 40, 20])][
    ["Graph_Name", "Graph_Portion", "Entropy_Arithmetic_Normalized"]
].pivot(index="Graph_Name", columns="Graph_Portion", values="Entropy_Arithmetic_Normalized")

df_target = df[df["Graph_Portion"] == 100][[
    "Graph_Name", "Entropy_AdamicAdar_Normalized", "Domain"
]].drop_duplicates().set_index("Graph_Name")

df_model = df_wide.merge(df_target, left_index=True, right_index=True)

# Fit Model 1
X1 = df_model[[100]].copy()
X1.columns = ["Entropy_Arithmetic_100"]
X1 = sm.add_constant(X1)
model1 = sm.OLS(df_model["Entropy_AdamicAdar_Normalized"], X1).fit()
df_model["Pred_Model_1"] = model1.predict(X1)

# Fit Model 5
X5 = df_model[[100, 80, 60, 40, 20]].copy()
X5.columns = [f"Entropy_Arithmetic_{p}" for p in [100, 80, 60, 40, 20]]
X5 = sm.add_constant(X5)
model5 = sm.OLS(df_model["Entropy_AdamicAdar_Normalized"], X5).fit()
df_model["Pred_Model_5"] = model5.predict(X5)

# Plot: predicted vs actual, colored by domain
plt.figure(figsize=(12, 5))

# Model 1
plt.subplot(1, 2, 1)
sns.scatterplot(
    data=df_model,
    x="Entropy_AdamicAdar_Normalized",
    y="Pred_Model_1",
    hue="Domain",
    palette="tab10"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("Model 1: Prediction vs Actual")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend(title="Domain")
plt.grid(True)

# Model 5
plt.subplot(1, 2, 2)
sns.scatterplot(
    data=df_model,
    x="Entropy_AdamicAdar_Normalized",
    y="Pred_Model_5",
    hue="Domain",
    palette="tab10"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("Model 5: Prediction vs Actual")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.legend([],[], frameon=False)  # Remove duplicate legend
plt.grid(True)

# plt.suptitle("Predicted vs Actual Entropy by Model and Domain")
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


### Residual Analysis

To further assess the model fit across network families, Figure A2 presents a residual 
analysis for both models. Model 1 exhibits pronounced domain-specific biases, with 
systematic underestimation or overestimation in several families, especially for 
economic and social networks. In contrast, residuals from Model 5 are more 
symmetrically distributed around zero and less variable, suggesting a significant 
reduction in systematic error.

**Fig. A2:** Distribution of residuals for predicted Adamic-Adar entropy across four 
network domains. (a) Model 1 shows high residual variance and domain-specific bias. 
(b) Model 5 (multiscale regression) substantially reduces residual variance compared 
to the single-scale model, with residuals more symmetrically distributed around zero 
across all domains. The red dashed line indicates zero residual.

These results show that structural entropy across scales contains predictive 
information that single-scale metrics miss. Multiscale entropy provides a more 
thorough characterization of network topology, extending beyond local compressibility. 
By doing so, multiscale entropy bridges the gap between compression and inference, 
providing a theoretically grounded framework for understanding the informational 
geometry of real-world networks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import shapiro
import statsmodels.api as sm

def plot_residuals_by_domain(X, y, df_metadata, model_name="Model", y_limits=None):
    """
    Fit an OLS model and plot residuals by domain.

    Parameters:
        X (DataFrame): Predictor variables.
        y (Series): Target variable.
        df_metadata (DataFrame): Must include 'Domain' and index matching X/y.
        model_name (str): Label for the plot title and summary.
        y_limits (tuple): Optional (ymin, ymax) to fix y-axis scale.
    """
    # Step 1: Fit model
    X_with_const = sm.add_constant(X)
    model = sm.OLS(y, X_with_const).fit()

    # Step 2: Compute residuals
    df_res = df_metadata.copy()
    df_res["Fitted"] = model.fittedvalues
    df_res["Residual"] = model.resid

    # Step 3: Summary statistics
    grouped = df_res.groupby("Domain")["Residual"].agg(["mean", "std", "count"]).reset_index()
    print(f"\n{model_name} - Residual Summary by Domain:")
    print(grouped)

    # Step 4: Boxplot with points
    plt.figure(figsize=(8, 6))
    sns.boxplot(
        data=df_res,
        x="Domain",
        y="Residual",
        hue="Domain",
        palette="pastel",
        fliersize=0,
        dodge=False,
        legend=False
    )
    sns.stripplot(
        data=df_res,
        x="Domain",
        y="Residual",
        color="black",
        jitter=0.0,
        size=4,
        alpha=0.5
    )

    plt.axhline(0, linestyle="--", color="red", linewidth=1)
    plt.ylabel("Residuals", fontsize=16)
    plt.xlabel("Domain", fontsize=16)
    plt.xticks(rotation=45, fontsize=14)
    plt.yticks(fontsize=14)
    if y_limits:
        plt.ylim(y_limits)
    plt.grid(True, axis="y", linestyle=":", linewidth=0.5)
    plt.tight_layout()
    # plt.title(f"{model_name}: Residuals by Domain", fontsize=16)
    plt.show()

    # Step 5: Normality test per domain
    print(f"\nShapiro-Wilk Normality Test for {model_name} (p < 0.05 indicates non-normal residuals):")
    for domain, group in df_res.groupby("Domain"):
        stat, p = shapiro(group["Residual"])
        print(f"  {domain:15s} --> p = {p:.4f}")


In [ ]:
# For Model 1
df_model_1 = df[df["Graph_Portion"] == 100]
X1 = df_model_1[["Entropy_Arithmetic_Normalized"]]
y = df_model_1["Entropy_AdamicAdar_Normalized"]
plot_residuals_by_domain(X1, y, df_model_1[["Domain"]], model_name="Model 1", y_limits=(-0.4, 0.4))

# For Model 5
df_wide = df[df["Graph_Portion"].isin([100, 80, 60, 40, 20])][
    ["Graph_Name", "Graph_Portion", "Entropy_Arithmetic_Normalized"]
].pivot(index="Graph_Name", columns="Graph_Portion", values="Entropy_Arithmetic_Normalized")

df_target = df[df["Graph_Portion"] == 100][["Graph_Name", "Entropy_AdamicAdar_Normalized", "Domain"]].set_index("Graph_Name")

df_model_5 = df_wide.merge(df_target, left_index=True, right_index=True)
X5 = df_model_5[[100, 80, 60, 40, 20]]
X5.columns = [f"Entropy_Arithmetic_{p}" for p in [100, 80, 60, 40, 20]]
y = df_model_5["Entropy_AdamicAdar_Normalized"]

plot_residuals_by_domain(X5, y, df_model_5[["Domain"]], model_name="Model 5", y_limits=(-0.4, 0.4))
